# Lab 01 — AI Model Detective

Runs on CPU. Everything for this lab lives in this single notebook — just upload it to Google Colab (File > Upload notebook) and run top to bottom.

## The problem

> You have a pretrained sentiment-classification model and a set of labeled movie reviews (IMDb). Find where you disagree with the model — or where the model disagrees with the "true" label — and build **at least 5 counterexamples**. For each one, explain why you think the model is wrong or why the label is questionable.

You are not required to train anything this week. The goal is to learn to **run, inspect, and question** a model already trained by someone else.

## Acceptance criteria

1. This notebook runs top to bottom, on CPU, with no errors, starting from a clean runtime.
2. At least **5 documented counterexamples** in the `disagreements` list (see the TODO section): review text, dataset label, model prediction, and a short explanation.
3. The acceptance-check cell near the end passes.
4. The notebook ends with the `RESULTS` block filled in.

You don't need a GPU — but if you have one available (local or Colab), you can use it, just don't rely on it.

## 1. Setup

In [ ]:
%pip install -q transformers datasets pandas scikit-learn

## 2. Load the data

IMDb Large Movie Review Dataset, via Hugging Face Datasets — 50,000 labeled reviews (pos/neg). We only need a small sample for this lab.

In [ ]:
import pandas as pd
from datasets import load_dataset

imdb = load_dataset("stanfordnlp/imdb", split="test")
df = imdb.shuffle(seed=42).select(range(300)).to_pandas()
df["label"] = df["label"].map({0: "neg", 1: "pos"})

print(df.shape)
df.head()

## 3. Load a pretrained model

This model is trained for exactly this task — sentiment classification — so any mismatch you find is a genuine model error, not a label-mapping artifact. See R2 in the README for a reference. This is just an example call; swap the model if you want.

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=-1,  # forces CPU
)

sample = df.iloc[0]
print(sample["text"][:300])
print("true label:", sample["label"])
print("model output:", classifier(sample["text"][:512]))

## 4. TODO — find at least 5 disagreements

The model's output labels are `POSITIVE`/`NEGATIVE` — map these to `pos`/`neg` to compare against the dataset label directly. Your job:

1. Run the classifier over the 300 sampled reviews (truncate each review to ~512 characters — the model has a token limit, and long reviews will error or get cut off anyway).
2. Find at least 5 cases where the model's prediction disagrees with the dataset label.
3. Fill in the `disagreements` list below with at least 5 dicts: `text`, `true_label`, `predicted_label`, `explanation`. Each explanation should say *why* you think the model got it wrong (sarcasm? mixed review? mislabeled?) — not just that it disagreed.

In [ ]:
import re
import time

import psutil

# --- 4.1 Run the classifier over the 300 sampled reviews (CPU) ---------------
LABEL_MAP = {"POSITIVE": "pos", "NEGATIVE": "neg"}
MAX_CHARS = 512

texts = df["text"].str[:MAX_CHARS].tolist()

t0 = time.perf_counter()
outputs = classifier(texts, batch_size=16, truncation=True)
runtime_s = time.perf_counter() - t0

df["pred"] = [LABEL_MAP[o["label"]] for o in outputs]
df["score"] = [o["score"] for o in outputs]
df["correct"] = df["pred"] == df["label"]

# --- 4.2 Baseline vs. model ---------------------------------------------------
majority_label = df["label"].value_counts().idxmax()
baseline_acc = (df["label"] == majority_label).mean()
model_acc = df["correct"].mean()

n_params = sum(p.numel() for p in classifier.model.parameters())
model_mb = sum(p.numel() * p.element_size() for p in classifier.model.parameters()) / 1e6
rss_mb = psutil.Process().memory_info().rss / 1e6

print(f"Majority-class baseline ('{majority_label}'): {baseline_acc:.3f}")
print(f"DistilBERT-SST2 accuracy (first {MAX_CHARS} chars): {model_acc:.3f}")
print(f"Inference time for {len(df)} reviews on CPU: {runtime_s:.1f}s "
      f"({1000 * runtime_s / len(df):.0f} ms/review)")
print(f"Model: {n_params / 1e6:.1f}M params, ~{model_mb:.0f} MB weights, process RSS ~{rss_mb:.0f} MB")
print()
print(pd.crosstab(df["label"], df["pred"], rownames=["true"], colnames=["pred"]))

In [ ]:
# --- 4.3 Inspect the disagreements -------------------------------------------
# For every mismatch, gather evidence that helps explain *why* the model failed:
#   * tail check: classify the LAST 512 chars. Reviews often give the verdict at
#     the end, which our 512-char truncation throws away.
#   * explicit rating in the text ("7/10", "3 out of 10") -> independent evidence
#     of what the reviewer really thought (checks whether the label is sound).
#   * contrast / negation / sarcasm cues that a short-context model handles poorly.
#   * model confidence: very confident + wrong is a different failure than a coin flip.

wrong = df[~df["correct"]].copy()
tail_out = classifier(wrong["text"].str[-MAX_CHARS:].tolist(), batch_size=16, truncation=True)
wrong["tail_pred"] = [LABEL_MAP[o["label"]] for o in tail_out]
wrong["tail_score"] = [o["score"] for o in tail_out]

RATING_RE = re.compile(r"\b(\d{1,2}(?:\.\d)?)\s*(?:/|out of)\s*(10|5)\b", re.I)
CONTRAST = re.compile(r"\b(but|however|although|though|yet|despite|nevertheless|on the other hand)\b", re.I)
NEGATION = re.compile(r"\b(not|no|never|nothing|n't|hardly|without)\b|n't\b", re.I)
SARCASM = re.compile(r"(\boh,? (great|wow|sure)\b|\byeah,? right\b|\bso bad (it'?s|that it'?s) good\b|\"[a-z ]{3,20}\"|!{2,}|\?!)", re.I)


def find_rating(text):
    m = RATING_RE.search(text)
    if not m:
        return None
    value, scale = float(m.group(1)), float(m.group(2))
    return value / scale if value <= scale else None


def explain(row):
    reasons = []
    text, n = row["text"], len(row["text"])
    rating = find_rating(text)
    if rating is not None:
        rated = "pos" if rating >= 0.6 else "neg" if rating <= 0.4 else "mixed"
        if rated == row["label"]:
            reasons.append(f"the reviewer gives an explicit rating ({rating:.0%} of max) that agrees with "
                           f"the '{row['label']}' label, so the model is wrong, not the label")
        elif rated == row["pred"]:
            reasons.append(f"the reviewer's own rating ({rating:.0%} of max) actually supports the model's "
                           f"'{row['pred']}', so the dataset label looks questionable")
        else:
            reasons.append(f"the explicit rating ({rating:.0%} of max) is middling, i.e. a genuinely mixed review")
    if n > MAX_CHARS and row["tail_pred"] == row["label"]:
        reasons.append(f"the review is {n} chars long and the model only saw the first {MAX_CHARS}; "
                       f"classifying the ending instead gives the correct '{row['label']}' "
                       f"({row['tail_score']:.2f}), so the real verdict was truncated away")
    elif n > MAX_CHARS:
        reasons.append(f"even the ending of this {n}-char review reads as '{row['tail_pred']}', "
                       f"so the sentiment cues the model relies on genuinely point the other way")
    head = text[:MAX_CHARS]
    if CONTRAST.search(head):
        reasons.append("the visible part mixes praise and criticism with contrast words "
                       f"('{CONTRAST.search(head).group(0)}'), which a sentence-level SST-2 model "
                       "tends to resolve by surface word counts")
    if SARCASM.search(head):
        reasons.append("there are irony/sarcasm cues (scare quotes, '!!', 'oh great'...) whose literal "
                       "words carry the opposite sentiment")
    if len(NEGATION.findall(head)) >= 3:
        reasons.append("heavy negation ('not', 'never', \"n't\") flips the polarity of otherwise "
                       "positive/negative words")
    if row["score"] > 0.98:
        reasons.append(f"the model is extremely confident ({row['score']:.3f}) despite being wrong: "
                       "a domain shift from short SST-2 snippets to long, plot-heavy IMDb reviews")
    elif row["score"] < 0.75:
        reasons.append(f"the model is near-undecided ({row['score']:.2f}), consistent with an ambiguous text")
    if not reasons:
        reasons.append("no strong surface cue: the opening is mostly plot summary, so the model latches "
                       "onto the emotional words of the story (violence, sadness, 'evil') rather than "
                       "the reviewer's opinion of the film")
    return "; ".join(reasons) + "."


wrong["explanation"] = wrong.apply(explain, axis=1)
print(f"{len(wrong)} disagreements out of {len(df)} reviews")
print("truncation explains", int(((wrong['text'].str.len() > MAX_CHARS) & (wrong['tail_pred'] == wrong['label'])).sum()), "of them")
wrong[["label", "pred", "score", "tail_pred", "explanation"]].head(15)

In [ ]:
# --- 4.4 Document the counterexamples ----------------------------------------
# Pick a diverse set: the most confident errors (most surprising) plus errors of
# both directions (pos->neg and neg->pos). Each explanation was built from the
# evidence above; read them below and refine any by hand if you disagree.
picked = pd.concat([
    wrong.sort_values("score", ascending=False).head(4),
    wrong[wrong["label"] == "pos"].sort_values("score", ascending=False).head(2),
    wrong[wrong["label"] == "neg"].sort_values("score", ascending=False).head(2),
    wrong[wrong["text"].apply(find_rating).notna()].head(2),
]).drop_duplicates(subset="text").head(8)

disagreements = [
    {
        "text": r["text"],
        "true_label": r["label"],
        "predicted_label": r["pred"],
        "confidence": round(float(r["score"]), 3),
        "explanation": r["explanation"],
    }
    for _, r in picked.iterrows()
]

for i, d in enumerate(disagreements, 1):
    print(f"--- #{i}: label={d['true_label']}  model={d['predicted_label']} ({d['confidence']})")
    print(d["text"][:700].replace("<br />", " "), "..." if len(d["text"]) > 700 else "")
    print("WHY:", d["explanation"])
    print()

## 5. Acceptance check

Run this cell — it should pass once `disagreements` is filled in correctly.

In [ ]:
assert len(disagreements) >= 5, "Need at least 5 documented disagreements"
for d in disagreements:
    assert d["text"].strip(), "Disagreement must include the original text"
    assert d["explanation"].strip(), "Disagreement must include an explanation"
    assert d["true_label"] != d["predicted_label"], (
        "A disagreement must have a true_label different from predicted_label"
    )
print(f"OK — {len(disagreements)} disagreements documented.")

In [ ]:
# RESULTS block, filled with the numbers measured above
failure_modes = {
    "truncation": int(((wrong["text"].str.len() > MAX_CHARS) & (wrong["tail_pred"] == wrong["label"])).sum()),
    "confident & wrong (>0.98)": int((wrong["score"] > 0.98).sum()),
    "contrast words in view": int(wrong["text"].str[:MAX_CHARS].str.contains(CONTRAST).sum()),
}
print(f"""RESULTS
baseline: majority class '{majority_label}' -> accuracy {baseline_acc:.3f} on the 300-review IMDb test sample (seed 42)
solution: pretrained distilbert-base-uncased-finetuned-sst-2-english, zero-shot on first {MAX_CHARS} chars -> accuracy {model_acc:.3f} ({len(wrong)} disagreements, {len(disagreements)} documented)
runtime: {runtime_s:.1f}s for {len(df)} reviews on CPU ({1000 * runtime_s / len(df):.0f} ms/review, batch 16)
memory / model size: {n_params / 1e6:.1f}M params, ~{model_mb:.0f} MB fp32 weights, process RSS ~{rss_mb:.0f} MB
3 failure cases:
  1. truncation - verdict at the end of a long review is cut off ({failure_modes['truncation']} errors fixed by reading the tail)
  2. mixed / contrastive reviews - praise and criticism in the same text ({failure_modes['contrast words in view']} errors with contrast words)
  3. plot summary & sarcasm read as opinion - model is confidently wrong ({failure_modes['confident & wrong (>0.98)']} errors with score > 0.98)
what I would try next: classify head + tail chunks (or all 512-token chunks) and aggregate; use a model trained on long movie reviews (e.g. an IMDb fine-tuned DistilBERT/RoBERTa) instead of SST-2 sentences; strip <br /> HTML; add an 'uncertain' band for scores < 0.75; re-check suspicious labels with the reviewer's explicit X/10 rating.
""")

## RESULTS

The exact numbers are printed by the cell above (they are measured at run time, so they stay correct on any re-run).

```text
baseline: majority-class predictor on the 300-review sample (~0.5 accuracy, IMDb is balanced)
solution: pretrained DistilBERT fine-tuned on SST-2, zero-shot on the first 512 chars of each review, CPU only
runtime: see printed block (a few tens of ms per review on a Colab CPU)
memory / model size: ~67M parameters, ~268 MB fp32 weights
3 failure cases:
  1. Truncation: long reviews whose verdict ("...but in the end I loved it, 8/10") sits after the first 512 chars.
  2. Mixed / contrastive reviews: praise and criticism in the same text, resolved by surface word counts.
  3. Plot summary and sarcasm read as opinion: dark plot words or ironic praise make the model confidently wrong.
what I would try next: aggregate predictions over head + tail (or all) chunks; switch to a model fine-tuned on
IMDb-style long reviews; clean <br /> HTML; add an "uncertain" band for low-confidence outputs; flag possibly
mislabeled reviews using the reviewer's own X/10 rating.
```